## **Preprocessing of Dataset 1**


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
import re

# Load the original dataset
df = pd.read_csv(r"C:\Users\nghui\FYP\EBT-main\data\Dataset 1\depression_dataset_reddit_cleaned.csv")

print("Original data sample:")
print(df.head())

# Check for missing values
print("\nChecking for missing values:")
print(df.isnull().sum())

# Drop rows with missing clean_text
df = df.dropna(subset=["clean_text"])

# Remove duplicates based on clean_text
print("\nChecking for duplicated texts:")
dup_count = df.duplicated(subset=["clean_text"]).sum()
print(f"Found {dup_count} duplicated rows.")

if dup_count > 0:
    df = df.drop_duplicates(subset=["clean_text"]).reset_index(drop=True)
    print(f"Removed {dup_count} duplicate texts. New dataset size: {len(df)}")
else:
    print("No duplicated texts found.")

# Text cleaning function: remove URLs, mentions, hashtags, HTML tags, and all special or non-ASCII symbols
def clean_text(text):
    text = re.sub(r"http\S+|www\S+", "", text)         # remove URLs
    text = re.sub(r"<.*?>", "", text)                  # remove HTML tags
    text = re.sub(r"@\w+", "", text)                   # remove @user mentions
    text = re.sub(r"#\w+", "", text)                   # remove hashtags
    text = re.sub(r"[^\x00-\x7F]+", "", text)          # remove emojis and non-ASCII symbols
    text = re.sub(r"[^\w\s.,!?']", "", text)           # remove special characters and number symbols like ①②③
    text = re.sub(r"\s+", " ", text).strip()           # normalize extra spaces
    return text

df["clean_text"] = df["clean_text"].astype(str).apply(clean_text)

# Shuffle dataset to ensure random distribution
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Check label distribution
print("\nLabel distribution before split:")
print(df["is_depression"].value_counts())

# Split dataset into 80% train, 10% validation, 10% test
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["is_depression"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["is_depression"]
)

# Save processed datasets
train_df.to_csv(r"C:\Users\nghui\FYP\EBT-main\data\Dataset 1\train.csv", index=False)
val_df.to_csv(r"C:\Users\nghui\FYP\EBT-main\data\Dataset 1\val.csv", index=False)
test_df.to_csv(r"C:\Users\nghui\FYP\EBT-main\data\Dataset 1\test.csv", index=False)

# Print summary
print("\nData successfully cleaned and split into train/val/test sets.")

print("\nTrain set label distribution:")
print(train_df["is_depression"].value_counts(normalize=True))

print("\nValidation set label distribution:")
print(val_df["is_depression"].value_counts(normalize=True))

print("\nTest set label distribution:")
print(test_df["is_depression"].value_counts(normalize=True))

print(f"\nTotal samples: {len(df)} | Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


Original data sample:
                                          clean_text  is_depression
0  we understand that most people who reply immed...              1
1  welcome to r depression s check in post a plac...              1
2  anyone else instead of sleeping more when depr...              1
3  i ve kind of stuffed around a lot in my life d...              1
4  sleep is my greatest and most comforting escap...              1

Checking for missing values:
clean_text       0
is_depression    0
dtype: int64

Checking for duplicated texts:
Found 81 duplicated rows.
Removed 81 duplicate texts. New dataset size: 7650

Label distribution before split:
is_depression
0    3889
1    3761
Name: count, dtype: int64

Data successfully cleaned and split into train/val/test sets.

Train set label distribution:
is_depression
0    0.508333
1    0.491667
Name: proportion, dtype: float64

Validation set label distribution:
is_depression
0    0.508497
1    0.491503
Name: proportion, dtype: float64

Test 

## **Preprocessing of Dataset 2**


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
import re
import json
from collections import OrderedDict
import unicodedata

# Paths
csv_in = r"C:\Users\nghui\FYP\EBT-main\data\Dataset 2\Combined_Data.csv"
out_train = r"C:\Users\nghui\FYP\EBT-main\data\Dataset 2\train.csv"
out_val = r"C:\Users\nghui\FYP\EBT-main\data\Dataset 2\val.csv"
out_test = r"C:\Users\nghui\FYP\EBT-main\data\Dataset 2\test.csv"

# Load
df = pd.read_csv(csv_in, encoding="utf-8")
print("Original data sample:")
print(df.head())

print("\nChecking for missing values:")
print(df.isnull().sum())

# Remove Unnamed cols (keep if you want)
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Ensure no NaN and force string type
df['statement'] = df['statement'].fillna("").astype(str)

def clean_statement_conservative(text):
    # remove control / unprintable chars
    text = "".join(ch for ch in text if unicodedata.category(ch)[0] != "C")
    # remove URLs, html tags, mentions, hashtags
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    # remove explicit control ascii ranges
    text = re.sub(r"[\x00-\x1f\x7f-\x9f]", "", text)
    # remove unusual characters but keep common punctuation and multilingual chars
    text = re.sub(r"[^\w\s\.,!?'\-—…，。！？：；（）《》、]+", "", text)
    # collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['statement'] = df['statement'].apply(clean_statement_conservative)

# Detect statements that are effectively empty / meaningless
def is_empty_statement(text, min_chars=1):
    # remove letters/digits/Chinese chars to check meaningful content length
    core = re.sub(r'[^a-zA-Z0-9\u4e00-\u9fff]', '', text)
    return len(core.strip()) < min_chars

# Set threshold: minimum meaningful characters (letters/digits/Chinese). 
MIN_MEANINGFUL_CHARS = 1

df['__is_empty__'] = df['statement'].apply(lambda x: is_empty_statement(x, min_chars=MIN_MEANINGFUL_CHARS))
empty_count = df['__is_empty__'].sum()
print(f"\nRows that are empty/meaningless after cleaning: {empty_count}")

# Remove empty rows (recommended). If you prefer to replace with UNK, see note below.
df = df[df['__is_empty__'] == False].reset_index(drop=True)
df = df.drop(columns=['__is_empty__'], errors='ignore')


# Remove exact duplicate statements (after cleaning)
dup_before = len(df)
df = df.drop_duplicates(subset=['statement']).reset_index(drop=True)
dup_after = len(df)
print(f"Removed {dup_before - dup_after} duplicate statements. Remaining rows: {dup_after}")

df["status"] = df["status"].astype(str).str.strip().str.lower()
print("\nStatus distribution (original labels):")
print(df["status"].value_counts())

# Shuffle dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Prepare stratify: only use stratify if both classes exist
if df["status"].nunique() > 1:
    stratify_col = df["status"]
else:
    stratify_col = None
    print("\nWarning: only one label present — stratified split disabled.")

# Split using numeric labels for stratify
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=stratify_col
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["status"] if stratify_col is not None else None
)
train_df.to_csv(out_train, index=False)
val_df.to_csv(out_val, index=False)
test_df.to_csv(out_test, index=False)

print("\nData successfully cleaned and split into train/val/test sets.")
print("\nTrain set label distribution:")
print(train_df["status"].value_counts(normalize=True))
print("\nValidation set label distribution:")
print(val_df["status"].value_counts(normalize=True))
print("\nTest set label distribution:")
print(test_df["status"].value_counts(normalize=True))
print(f"\nTotal samples: {len(df)} | Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Original data sample:
                                           statement  status
0                                         oh my gosh       1
1  trouble sleeping, confused mind, restless hear...       1
2  All wrong, back off dear, forward doubt. Stay ...       1
3  I've shifted my focus to something else but I'...       1
4  I'm restless and restless, it's been a month n...       1

Checking for missing values:
statement    362
status         0
dtype: int64

Rows that are empty/meaningless after cleaning: 384
Removed 1626 duplicate statements. Remaining rows: 51033

Status distribution (original labels):
status
1    35018
0    16015
Name: count, dtype: int64

Data successfully cleaned and split into train/val/test sets.

Train set label distribution:
status
1    0.68618
0    0.31382
Name: proportion, dtype: float64

Validation set label distribution:
status
1    0.686263
0    0.313737
Name: proportion, dtype: float64

Test set label distribution:
status
1    0.686129
0    0.313871
N